# Scoring the 500-pair v3 benchmark

Four tools on the same 500 held-out pairs: **savant v4/esm650**,
**savant v4/esm8**, **Boltz-2**, **PRODIGY-LIG**, plus a constant baseline.

The question this notebook is built to answer is narrow on purpose: **in what
context is savant as good as or better than Boltz-2?** Not "which tool is best" -
that depends entirely on which of the two abilities below you are buying.

The benchmark is **ligand-novel** (ECFP4 Tanimoto < 0.50 to anything in training)
and **temporal** (2025+, absent from ChEMBL v34). It is hard on purpose.

**It is not protein-novel, and that is deliberate.** `build_benchmark.py` does not gate
proteins - input.csv covers essentially every well-measured target, so requiring
novel proteins would have left only sparsely measured ones. All 20 benchmark
targets appear in training with other ligands (19 of 20 have a median of ~1,169
each). So this measures **ranking novel chemistry against an established
target**, not generalisation to a new target. That is the realistic screening and
lead-optimisation setting, but it must not be described as zero-shot on the
target, and the within-target scores below are not zero-shot numbers.

## Five test cases, because one number hides the story

A single pooled correlation blends two different abilities in whatever ratio the
benchmark happens to have. Here that ratio is 48% between-target / 52%
within-target, so a pooled score is roughly half a test of "which protein binds
tighter" and half a test of "which ligand binds tighter". A model can be
excellent at one and blind to the other and land mid-table.

| test | question | grouping |
|---|---|---|
| **A. pooled** | absolute pKd across everything | none |
| **B. one protein, many ligands** | rank a compound series | by `uniprot_id` |
| **C. one ligand, many proteins** | selectivity across a panel | by ligand `skeleton` |
| **D. protein taxonomy** | where by target family | curated family map |
| **E. ligand taxonomy** | where by chemotype | RDKit SMARTS classes |

Test B is the lead-optimisation case. Test C is the selectivity case and needs no
extra data: 86 benchmark ligands were measured against 2-4 proteins each.

## The one baseline that must appear in every table

**Constant mean** predicts the benchmark average for every pair. In pooled RMSE
this is a genuinely hard baseline, and on this benchmark most real tools lose to
it. Any RMSE claim that does not clear this line is not a result.

It is not a competitor - it is the floor that says whether a number means
anything.

In [ ]:
import json, sys, warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import pearsonr, spearmanr

warnings.filterwarnings("ignore", category=RuntimeWarning)

EVAL   = Path.cwd()                                  # run from benchmark/eval/
OUT    = EVAL / "output"
REPO   = EVAL.parents[2]
## savant is a package at <repo>/savant, so the REPO root goes on sys.path.
SAVANT_PATH = REPO

## models_full is the model of record and is scored as "Savant". work_full_v2 is
## an earlier raw-label build that also carried a classifier; it lives only on the
## cluster, is NOT part of this branch, and nothing in the paper reports it. The
## dict is filtered by p.exists() below, so a missing directory is skipped rather
## than raising - set INCLUDE_PREVIOUS = False to drop it even when present.
MODEL_SETS = {
    "":    REPO / "ML_model" / "models_full",   # averaged labels, regressor only
    "v2_": REPO / "ML_model" / "work_full_v2",   # raw labels, with classifier
}
INCLUDE_PREVIOUS = True

## Input: the merged benchmark carrying every tool's columns.
##
## Pick the MOST RECENTLY MODIFIED candidate, not the first that exists. Taking
## the first in a fixed order silently prefers a stale file: 06_merge_scores.py
## writes <benchmark stem>_scored.csv, so a newly added tool lands there while an
## older v3bench_scored.csv from a previous run still sits in the same folder.
## The notebook then reports the new tool as "absent or empty" when the column is
## simply in a file it never opened.
##
## Set BENCH_SCORED_CSV to override.
import os
CANDIDATES = [OUT / "savant_benchmark_scored.csv",   # what 06_merge_scores writes
              OUT / "v3bench_scored.csv",
              OUT / "bench_scored.csv",                 # this notebook's own output
              EVAL.parent / "output" / "savant_benchmark.csv"]

_override = os.environ.get("BENCH_SCORED_CSV")
if _override:
    BENCH = Path(_override)
    if not BENCH.exists():
        raise SystemExit(f"BENCH_SCORED_CSV={BENCH} does not exist")
else:
    present = [p for p in CANDIDATES if p.exists()]
    if not present:
        raise SystemExit(f"no benchmark csv found; looked for "
                         f"{[str(p) for p in CANDIDATES]}")
    BENCH = max(present, key=lambda p: p.stat().st_mtime)
    for p in sorted(present, key=lambda p: p.stat().st_mtime, reverse=True):
        stamp = pd.Timestamp(p.stat().st_mtime, unit="s").strftime("%Y-%m-%d %H:%M")
        print(f"  {'USING ' if p == BENCH else '      '}{p.name:38s} {stamp}")

BINDER_CUTS  = [6.0, 7.0]      # 6.0 = trained cut; 7.0 = balanced on this set
MIN_TARGET_N = 5               # targets below this are unusable for ranking
MIN_STRATUM  = 20              # rows below this: report n, do not rank
N_BOOT       = 2000
SEED         = 1337

OUT.mkdir(exist_ok=True)
df = pd.read_csv(BENCH)
print(f"{BENCH.name}: {len(df)} rows, {df.uniprot_id.nunique()} targets, "
      f"{df.skeleton.nunique()} ligands")
TRUTH = "neg_log10_affinity_M"
print(f"truth pKd: mean {df[TRUTH].mean():.2f}  sd {df[TRUTH].std():.2f}  "
      f"range {df[TRUTH].min():.2f}-{df[TRUTH].max():.2f}")

## Score the savant variants

Scoring goes through `savant.Savant`, so the features here are built by the
same code that builds them in production - not by a copy that can drift.
`Savant.features` enforces both the per-block widths and the total, and the block
order lives in one place.

**Features are built once per variant and reused across model generations.** The
two generations were trained on the same feature blocks with the same RDKit
column list and the same encoder, so their matrices are identical; rebuilding
would mean pushing 500 sequences through ESM2-650M twice for the same numbers.
The cell verifies the layouts actually match before reusing, and rebuilds if they
do not.

The current models have no classifier, so their `p_binder` column comes back
all-NaN and is dropped automatically from the classification tables further down.
That is intentional - the regressor's predicted pKd is what gets thresholded.

If a generation's bundles are missing it is skipped, and any savant columns
already in the CSV are used instead.

In [ ]:
sys.path.insert(0, str(SAVANT_PATH))
from savant import Savant

SETS = {t: p for t, p in MODEL_SETS.items()
        if p.exists() and (t == "" or INCLUDE_PREVIOUS)}
if SETS:
    for tag, p in SETS.items():
        print(f"model set {tag or 'current':4s} -> {p}")
else:
    print("no model directories found; falling back to savant columns in the csv")

VARIANTS = {}
for name in ("esm650", "esm8"):
    loaded = {}
    for tag, root in SETS.items():
        if (root / f"savant_full_{name}.pkl").exists() or (root / name).exists():
            loaded[tag] = Savant(model=name, models_dir=root)
    if not loaded:
        print(f"skip {name}: no bundles in any model set")
        continue

    ref = next(iter(loaded.values()))
    X = ref.features(df.target_sequence.tolist(), df.ligand_smiles.tolist())
    for tag, sv in loaded.items():
        same = (sv.n_features_in_ == ref.n_features_in_
                and list(sv.rdkit_columns) == list(ref.rdkit_columns)
                and sv.esm_model == ref.esm_model)
        Xi = X if same else sv.features(df.target_sequence.tolist(),
                                        df.ligand_smiles.tolist())
        if not same:
            print(f"  {tag or 'current'}/{name}: layout differs, features rebuilt")
        out = sv.predict_from(Xi)
        df[f"savant_{tag}{name}_pk"] = out.pKd.to_numpy()
        df[f"savant_{tag}{name}_p"]  = out.p_binder.to_numpy()
        VARIANTS[f"{tag}{name}"] = sv
        print(f"  {tag or 'current':4s}/{name}: {sv.n_features_in_} feats, "
              f"classifier={'yes' if sv.classifier is not None else 'no'}, "
              f"pKd mean {out.pKd.mean():.2f} sd {out.pKd.std():.2f}")
    del X

## The per-target baselines

The obvious competitor in this setting, and the first thing a reviewer will ask
about. The benchmark is **not** protein-novel: every target appears in training
with other ligands. So a chemist facing one of these targets would not reach for
a general model - they would fit a local model on that target's own measured
compounds. Two of those, both standard:

- **QSAR per-target** - ECFP4 counts + six cheap descriptors, one XGBoost per
  target. What a computational chemist builds in an afternoon.
- **Tanimoto kNN** - similarity-weighted mean of the k nearest known actives.
  Cruder, and in lead-op territory it is often hard to beat.

**They are trained on exactly the rows savant was trained on**, read from
`dataset_rows.csv` in the model directory and filtered to one target. That
matters: those rows already passed the ECFP4 Tanimoto < 0.50 benchmark gate, so
the baselines are held to the same novelty standard. Rebuilding them from the raw
`input.csv` would let them train on near-duplicates of benchmark ligands that
savant was forbidden, and the comparison would be rigged in their favour.

They are computed here, before the tool registry, so they appear in **every** test
below rather than in an isolated section. Test F then asks the question that
actually decides the framing: how does the gap scale with how much prior data the
target has?

A per-target model predicts absolute pKd, so it is legitimately comparable in the
pooled and between-target tables too - and it should do well there, since it
learns each target's scale directly.

In [ ]:
QSAR_MIN_TRAIN = 30               # targets below this get no baseline
QSAR_KNN_K     = 5

from rdkit import Chem, RDLogger
from rdkit.Chem import (Crippen, Descriptors, Lipinski, rdFingerprintGenerator,
                        rdMolDescriptors)
RDLogger.DisableLog("rdApp.*")
_gen = rdFingerprintGenerator.GetMorganGenerator(radius=2, fpSize=1024)

def qsar_features(smiles):
    '''(counts+descriptors, binary fp, parsed-ok mask). Bare molecule, no AddHs:
    this is a standalone baseline, not something that has to match savant's
    training columns, and skipping AddHs keeps ~40k ligands under a minute.'''
    fp, bits, desc, ok = [], [], [], []
    for s in smiles:
        m = Chem.MolFromSmiles(s)
        ok.append(m is not None)
        if m is None:
            continue
        fp.append(_gen.GetCountFingerprintAsNumPy(m))
        bits.append(_gen.GetFingerprintAsNumPy(m))
        desc.append([Descriptors.MolWt(m), Crippen.MolLogP(m),
                     rdMolDescriptors.CalcTPSA(m), Lipinski.NumHDonors(m),
                     Lipinski.NumHAcceptors(m), Lipinski.NumRotatableBonds(m)])
    X = np.hstack([np.asarray(fp, np.float32), np.asarray(desc, np.float32)])
    return X, np.asarray(bits, bool), np.asarray(ok)

def tanimoto(A, B):
    '''(n,m) Tanimoto between two sets of binary fingerprints.'''
    a, b = A.astype(np.float32), B.astype(np.float32)
    inter = a @ b.T
    return inter / (a.sum(1)[:, None] + b.sum(1)[None, :] - inter + 1e-9)

## The files do not all live in one directory. train_savant_full_dedup.ipynb
## writes only dataset_rows.csv into models_full and deliberately reuses
## work_full_v2's ligands.csv / proteins.csv, so that the row ids still line up
## with the cached feature blocks. Each file is therefore resolved independently,
## newest model set first, and the id ranges are checked below.
def _find(fname):
    for root in MODEL_SETS.values():
        if (Path(root) / fname).exists():
            return Path(root) / fname
    return None

QSAR_FILES = {f: _find(f) for f in ("dataset_rows.csv", "ligands.csv", "proteins.csv")}
df["qsar_pk"] = np.nan
df["knn_pk"] = np.nan
QSAR_N_TRAIN = {}

for f, p in QSAR_FILES.items():
    print(f"  {f:20s} {p if p else 'NOT FOUND'}")
missing = [f for f, p in QSAR_FILES.items() if p is None]

if missing:
    print(f"\nskip per-target baselines: could not find {missing}")
    print(f"searched: {[str(r) for r in MODEL_SETS.values()]}")
else:
    import xgboost as xgb
    rows_tr = pd.read_csv(QSAR_FILES["dataset_rows.csv"])
    ligs = pd.read_csv(QSAR_FILES["ligands.csv"])
    prots = pd.read_csv(QSAR_FILES["proteins.csv"])
    ## the rows must actually index into these tables - a mismatched pairing
    ## would silently train on the wrong molecules
    assert rows_tr.ligand_id.max() < len(ligs), "ligand_id out of range for ligands.csv"
    assert rows_tr.protein_id.max() < len(prots), "protein_id out of range for proteins.csv"
    seq2pid = dict(zip(prots.seq, prots.protein_id))
    lig_smiles = ligs.set_index("ligand_id").smiles

    df["_pid"] = df.target_sequence.map(seq2pid)
    print(f"benchmark targets found in the training set: "
          f"{df._pid.notna().groupby(df.uniprot_id).any().sum()} of "
          f"{df.uniprot_id.nunique()}")

    for uid, sub in df.groupby("uniprot_id"):
        pid = sub._pid.dropna().unique()
        if len(pid) == 0:
            QSAR_N_TRAIN[uid] = 0
            continue
        tr = rows_tr[rows_tr.protein_id == pid[0]]
        QSAR_N_TRAIN[uid] = len(tr)
        if len(tr) < QSAR_MIN_TRAIN:
            continue

        Xtr, Btr, ok_tr = qsar_features(lig_smiles.loc[tr.ligand_id].tolist())
        ytr = tr.neg_log10_affinity_M.to_numpy(np.float32)[ok_tr]
        Xte, Bte, ok_te = qsar_features(sub.ligand_smiles.tolist())
        idx = sub.index[ok_te]

        ## small internal split so the per-target model early-stops rather than
        ## running a fixed round count on 30 rows or 9,000
        n_val = max(1, int(0.2 * len(ytr)))
        perm = np.random.default_rng(SEED).permutation(len(ytr))
        vi, fi = perm[:n_val], perm[n_val:]
        booster = xgb.train(
            {"objective": "reg:squarederror", "eval_metric": "rmse",
             "max_depth": 6, "eta": 0.05, "subsample": 0.8,
             "colsample_bytree": 0.5, "min_child_weight": 3,
             "tree_method": "hist", "nthread": -1, "seed": SEED},
            xgb.DMatrix(Xtr[fi], label=ytr[fi]), num_boost_round=600,
            evals=[(xgb.DMatrix(Xtr[vi], label=ytr[vi]), "val")],
            early_stopping_rounds=30, verbose_eval=False)
        df.loc[idx, "qsar_pk"] = booster.inplace_predict(Xte)

        sim = tanimoto(Bte, Btr)                     # (n_test, n_train)
        k = min(QSAR_KNN_K, sim.shape[1])
        top = np.argpartition(-sim, k - 1, axis=1)[:, :k]
        w = np.take_along_axis(sim, top, 1)
        df.loc[idx, "knn_pk"] = ((w * ytr[top]).sum(1) / (w.sum(1) + 1e-9))

    n = pd.Series(QSAR_N_TRAIN)
    print(f"per-target training ligands: median {n.median():.0f}  "
          f"min {n.min()}  max {n.max()}")
    print(f"targets with a baseline: {df.qsar_pk.notna().groupby(df.uniprot_id).any().sum()}"
          f" (needed >= {QSAR_MIN_TRAIN} ligands)")
    print(f"benchmark rows scored: qsar {df.qsar_pk.notna().sum()}, "
          f"knn {df.knn_pk.notna().sum()} of {len(df)}")
    df.drop(columns=["_pid"], inplace=True)

## The tool registry

A tool is included only if its column exists and is not entirely empty - so a
missing PRODIGY+elec or PLAPT run drops out silently rather than producing a row
of NaN.

**PLAPT** (Rose et al., bioRxiv 2024.02.08.575577) is the closest published peer
to Savant: sequence + SMILES in, pKd out, no structure. It uses ProtBERT and
ChemBERTa embeddings with a small ONNX head, where Savant uses ESM2 and ChemBERTa
with gradient-boosted trees. Because it needs no pose, its coverage is not capped
by the Boltz stage - expect it on all 500 pairs where Boltz-2 reaches 496.

It was posted 2024-02-09, so the temporal gate (measurements from 2025 onward,
absent from ChEMBL v34) rules out contamination the same way it does for Boltz-2.
The ligand-similarity gate does **not** transfer: it stripped ligands within
Tanimoto 0.50 of a benchmark ligand from *Savant's* training set, not PLAPT's.
That biases the comparison against Savant, which is the conservative direction.

In [ ]:
df["constant_pk"] = df[TRUTH].mean()

CANDIDATE_TOOLS = {
    "Savant esm650":        "savant_esm650_pk",       # current: averaged labels
    "Savant esm8":          "savant_esm8_pk",
    "Savant-v2 esm650":     "savant_v2_esm650_pk",    # previous: raw labels
    "Savant-v2 esm8":       "savant_v2_esm8_pk",
    "Boltz-2":              "boltz_affinity_pic50",
    "PRODIGY-LIG":          "prodigy_pkd",
    "PRODIGY-LIG +elec":    "prodigy_pkd_elec",
    ## PLAPT: structure-free like Savant, so its coverage is not capped by
    ## whether Boltz-2 produced a pose. Already on the pKd scale.
    "PLAPT":                "plapt_pk",
    "QSAR per-target":      "qsar_pk",                # trained on the target's own data
    "Tanimoto kNN":         "knn_pk",
    "Constant mean":        "constant_pk",
}
TOOLS = {k: v for k, v in CANDIDATE_TOOLS.items()
         if v in df.columns and df[v].notna().any()}
BASELINES = {"Constant mean"}
MODELS = [k for k in TOOLS if k not in BASELINES]
REF = "Boltz-2" if "Boltz-2" in TOOLS else MODELS[0]

## The constant baseline has no within-target variation at all, so every
## within-target rank correlation it could produce is undefined rather than zero.
## It is excluded wherever a within-target number is reported and kept in the
## pooled table, where it is the floor that makes RMSE readable.
NO_WITHIN = BASELINES

## an all-NaN column means that generation shipped no classifier - dropped here
## rather than scored as if it had one
CLF_COLS = {"Savant esm650": "savant_esm650_p", "Savant esm8": "savant_esm8_p",
            "Savant-v2 esm650": "savant_v2_esm650_p",
            "Savant-v2 esm8": "savant_v2_esm8_p",
            "Boltz-2": "boltz_affinity_probability_binary"}
CLF_COLS = {k: v for k, v in CLF_COLS.items() if v in df.columns and df[v].notna().any()}

for k, v in TOOLS.items():
    print(f"  {k:22s} {v:36s} n={df[v].notna().sum()}")

## Distinguish the two reasons a tool drops out. "Column not in this file" almost
## always means the wrong CSV was loaded - the tool ran, but its column landed in
## a file this notebook did not open. "All-NaN" means the column is there and the
## tool genuinely produced nothing (or, for a regressor-only savant bundle, that
## there is deliberately no classifier probability).
absent = {k: v for k, v in CANDIDATE_TOOLS.items()
          if k not in TOOLS and v not in df.columns}
empty = {k: v for k, v in CANDIDATE_TOOLS.items()
         if k not in TOOLS and v in df.columns}
if absent:
    print(f"\ndropped - column not present in {BENCH.name}:")
    for k, v in absent.items():
        print(f"  {k:22s} {v}")
    print("  If a tool did run, its column is in a different CSV. Check the file "
          "list printed above, or set BENCH_SCORED_CSV.")
if empty:
    print("\ndropped - column present but entirely empty:")
    for k, v in empty.items():
        print(f"  {k:22s} {v}")
print(f"\nreference for paired tests: {REF}")

## Shared helpers

`grouped_rank` is the engine for tests B and C: it splits the rows by a key,
computes a per-group Spearman, and averages. `paired_bootstrap` resamples the
**groups** (not the rows) so the interval reflects the fact that 20 targets is a
small sample - resampling rows instead would report an interval several times too
narrow.

In [ ]:
def metrics(y, p):
    y, p = np.asarray(y, float), np.asarray(p, float)
    m = np.isfinite(y) & np.isfinite(p)
    y, p = y[m], p[m]
    if len(y) < 3 or p.std() == 0:
        return {"n": len(y)}
    err = p - y
    return {"n": len(y),
            "pearson":  pearsonr(y, p)[0],
            "spearman": spearmanr(y, p).correlation,
            "rmse":     float(np.sqrt((err ** 2).mean())),
            "mae":      float(np.abs(err).mean()),
            "bias":     float(err.mean()),
            "r2":       1 - (err ** 2).sum() / ((y - y.mean()) ** 2).sum(),
            "pred_std_ratio": float(p.std() / y.std())}


def grouped_rank(data, key, col, min_n=MIN_TARGET_N):
    '''Per-group Spearman between truth and one tool. Returns a Series by group.'''
    out = {}
    for k, s in data.groupby(key):
        s = s[np.isfinite(s[TRUTH]) & np.isfinite(s[col])]
        if len(s) < min_n or s[col].std() == 0:
            continue
        out[k] = spearmanr(s[TRUTH], s[col]).correlation
    return pd.Series(out, dtype=float).dropna()


def paired_bootstrap(a, b, n_boot=N_BOOT, seed=SEED):
    '''CI for mean(a - b) over the groups both scored. Resamples groups.'''
    common = a.index.intersection(b.index)
    if len(common) < 3:
        return {"n_groups": len(common)}
    d = (a[common] - b[common]).to_numpy()
    rng = np.random.default_rng(seed)
    draws = d[rng.integers(0, len(d), size=(n_boot, len(d)))].mean(axis=1)
    return {"n_groups": len(common), "mean_diff": float(d.mean()),
            "ci_low": float(np.percentile(draws, 2.5)),
            "ci_high": float(np.percentile(draws, 97.5)),
            "P_better": float((draws > 0).mean()),
            "wins": int((d > 0).sum())}


def table(rows, sort=None, asc=False):
    t = pd.DataFrame(rows).set_index("tool")
    return t.sort_values(sort, ascending=asc) if sort and sort in t else t

## Test A - pooled

Every pair, ignoring which target it belongs to.

Read `r2` and `rmse` against **Constant mean**, which by construction has
`r2 = 0` and `rmse` equal to the label standard deviation. A tool with negative
`r2` here is worse than predicting the average, and its RMSE is not a result no
matter how it compares to other tools' RMSE.

`pred_std_ratio` is the predicted spread over the true spread. Well below 1 means
the tool is hedging toward the middle - which lowers RMSE without adding any
information, and is exactly how a weak model wins an RMSE race.

In [ ]:
pooled = table([{"tool": k, **metrics(df[TRUTH], df[v])} for k, v in TOOLS.items()],
               sort="pearson")
display(pooled.round(3))

const_rmse = float(pooled.loc["Constant mean", "rmse"])
beat = pooled[(pooled.rmse < const_rmse) & (~pooled.index.isin(BASELINES))]
print(f"\nconstant-mean RMSE = {const_rmse:.3f}")
print(f"tools beating it: {list(beat.index) if len(beat) else 'NONE'}")
pooled.to_csv(OUT / "bench_A_pooled.csv")

### Where the pooled number comes from

Splitting each tool's pooled correlation into its two parts. **Between** is the
correlation of the 20 predicted target means against the true target means -
"does it know which protein binds tighter". **Within** centres both truth and
prediction on their target mean before correlating - "does it know which ligand
binds tighter, given the target".

These two columns explain most disagreements between the pooled table and test B.

In [ ]:
rows = []
for k, v in TOOLS.items():
    s = df[np.isfinite(df[TRUTH]) & np.isfinite(df[v])]
    if len(s) < 10:
        continue
    gt = s.groupby("uniprot_id")[TRUTH].transform("mean")
    gp = s.groupby("uniprot_id")[v].transform("mean")
    tm = s.groupby("uniprot_id").agg(t=(TRUTH, "mean"), p=(v, "mean"))
    within = np.nan if k in NO_WITHIN or (s[v] - gp).std() == 0 \
        else pearsonr(s[TRUTH] - gt, s[v] - gp)[0]
    rows.append({"tool": k, "n": len(s),
                 "between_r": pearsonr(tm.t, tm.p)[0] if tm.p.std() > 0 else np.nan,
                 "within_r": within,
                 "pooled_r": pearsonr(s[TRUTH], s[v])[0],
                 "target_mean_spread": float(tm.p.std())})
decomp = table(rows, sort="pooled_r")
display(decomp.round(3))
print(f"\ntrue target-mean spread (sd): "
      f"{df.groupby('uniprot_id')[TRUTH].mean().std():.3f}")
print("a tool whose target_mean_spread is far below this is guessing the same "
      "value for every target")
decomp.to_csv(OUT / "bench_A_decomposition.csv")

## Test B - one protein, many ligands

The lead-optimisation case: given a target, rank its compounds. Spearman inside
each target, averaged over targets. Targets with fewer than `MIN_TARGET_N` pairs
are dropped - a rank correlation over 3 points is noise.

The paired bootstrap resamples targets, so the interval answers "would this
ordering hold on a different draw of 20 targets", which is the question that
matters and the reason these intervals are wide.

In [ ]:
per_target = pd.DataFrame({k: grouped_rank(df, "uniprot_id", v)
                           for k, v in TOOLS.items() if k not in NO_WITHIN})
summary = pd.DataFrame({
    "n_targets": per_target.notna().sum(),
    "mean_spearman": per_target.mean(),
    "median_spearman": per_target.median(),
    "n_positive": (per_target > 0).sum(),
}).sort_values("mean_spearman", ascending=False)
display(summary.round(3))

from scipy.stats import binomtest, wilcoxon

def paired_tests(a, b):
    '''Bootstrap CI plus two distribution-free tests on the same paired values.

    The bootstrap CI and P_better describe the mean difference. Wilcoxon asks
    whether the differences are centred away from zero, and the sign test asks
    only how often one wins. They can disagree: a large mean difference driven by
    a few targets gives a suggestive bootstrap and a null sign test, which is a
    real distinction to report rather than to pick between.
    '''
    common = a.index.intersection(b.index)
    d = (a[common] - b[common]).dropna()
    out = paired_bootstrap(a, b)
    if len(d) >= 5:
        out["wilcoxon_p"] = float(wilcoxon(d).pvalue)
        out["sign_test_p"] = float(binomtest(int((d > 0).sum()), len(d), 0.5).pvalue)
    return out

vs = table([{"tool": k, **paired_tests(per_target[k], per_target[REF])}
            for k in per_target.columns if k != REF], sort="mean_diff")
print(f"\npaired vs {REF} (resampling targets):")
display(vs.round(3))
print("A CI spanning zero means parity, whatever the point estimate says.")

pairs = df.groupby("uniprot_id").size()
print(f"\ntargets: {len(pairs)}  usable (n>={MIN_TARGET_N}): {(pairs>=MIN_TARGET_N).sum()}"
      f"  median pairs/target: {pairs.median():.0f}")
per_target.to_csv(OUT / "bench_B_per_target.csv")
summary.to_csv(OUT / "bench_B_summary.csv")
vs.to_csv(OUT / "bench_B_vs_ref.csv")

## Test C - one ligand, many proteins

Selectivity. 86 benchmark ligands were measured against 2-4 proteins each - the
CA1/CA2/CA9/CA12 isoform panel, the vasopressin V1a/V2 pair, and a few others.
Predicting which isoform a compound prefers is a real medicinal-chemistry problem
and it is the between-target axis in its purest form.

Spearman degenerates at k=2 (it can only be +1 or -1), so the primary metric is
**pairwise sign accuracy**: over every (ligand, protein-pair) combination, does
the tool get the sign of the true difference right? Chance is 0.50. Pairs whose
true gap is under `MIN_GAP` are dropped - below the assay noise floor (~0.4-0.85
pKd, measured from replicate disagreement in the training data) the "correct"
answer is not reliably defined.

**Two versions of it, and the second is the real test.**

The raw version is largely answerable without looking at the ligand at all: if
CA2 binds most things more tightly than CA1, then "CA2 wins" is right most of the
time, for every compound. A predictor that only knew each target's average
affinity - no ligand information whatsoever - scores about **0.88** on the raw
metric, above every real tool. That is a property of the question, not a fault in
the tools, and it means the raw column mostly measures base rate.

The **corrected** version subtracts each target's mean from both truth and
prediction first, so the question becomes: does *this* compound prefer CA9 **more
than a typical compound does**? That is the actual selectivity question a med
chemist asks, and the base rate cannot answer it.

**Read the corrected column.** Keep the raw one only to see how much of a raw
score was base rate.

In [ ]:
from itertools import combinations

MIN_GAP = 0.5
k = df.groupby("skeleton").uniprot_id.nunique()
multi = df[df.skeleton.isin(k[k >= 2].index)].copy()
print(f"{k.ge(2).sum()} ligands across 2+ proteins, {len(multi)} rows")
print(f"proteins covered: {multi.uniprot_id.nunique()}")

SEL_TOOLS = {k: v for k, v in TOOLS.items() if k not in BASELINES}
CEN_TOOLS = SEL_TOOLS

## target-mean-centred copies: removes the "which protein binds tighter in
## general" component that the base rate alone answers
cen = multi.copy()
cen["_t"] = multi[TRUTH] - multi.groupby("uniprot_id")[TRUTH].transform("mean")
for name, col in CEN_TOOLS.items():
    cen[f"_p_{name}"] = multi[col] - multi.groupby("uniprot_id")[col].transform("mean")

def hit(pa, pb, gap):
    '''Did the tool get the sign right? NaN if it cannot answer.

    A predicted difference of exactly zero is not a wrong answer, it is no
    answer - counting it as a miss would score a constant predictor at 0.00 and
    make it look worse than chance when it simply has no signal here.
    '''
    if not (np.isfinite(pa) and np.isfinite(pb)) or pa == pb:
        return np.nan
    return np.sign(pa - pb) == np.sign(gap)

raw_recs, cen_recs, n_all = [], [], 0
for skel, s in cen.groupby("skeleton"):
    for i, j in combinations(range(len(s)), 2):
        a, b = s.iloc[i], s.iloc[j]
        n_all += 1
        gap, cgap = a[TRUTH] - b[TRUTH], a["_t"] - b["_t"]
        if abs(gap) >= MIN_GAP:
            raw_recs.append({"skeleton": skel, **{
                n: hit(a[c], b[c], gap) for n, c in SEL_TOOLS.items()}})
        if abs(cgap) >= MIN_GAP:
            cen_recs.append({"skeleton": skel, **{
                n: hit(a[f"_p_{n}"], b[f"_p_{n}"], cgap) for n in CEN_TOOLS}})

def sign_table(recs, label):
    t = pd.DataFrame(recs)
    cols = [c for c in t.columns if c != "skeleton"]
    ## bool-or-NaN columns land as object dtype, which makes .mean() return an
    ## object and the sqrt below fail; float is the honest dtype here anyway
    t[cols] = t[cols].astype(float)
    out = pd.DataFrame({f"n_{label}": t[cols].notna().sum(),
                        f"{label}_accuracy": t[cols].mean()})
    out[f"{label}_ci_halfwidth"] = 1.96 * np.sqrt(
        out[f"{label}_accuracy"] * (1 - out[f"{label}_accuracy"]) / out[f"n_{label}"])
    out[f"{label}_beats_chance"] = \
        out[f"{label}_accuracy"] - out[f"{label}_ci_halfwidth"] > 0.5
    return out

acc = sign_table(raw_recs, "raw").join(sign_table(cen_recs, "corrected"), how="left")
acc = acc.sort_values("corrected_accuracy", ascending=False, na_position="last")
print(f"{n_all} ligand/protein-pair combinations total")
print(f"  raw:       {len(raw_recs)} with a true gap >= {MIN_GAP} pKd")
print(f"  corrected: {len(cen_recs)} with a centred gap >= {MIN_GAP} pKd")
display(acc.round(3))
print("\ncorrected_accuracy is the selectivity result; chance is 0.50.")
print("beats_chance False for every tool would be a real finding about the field, "
      "not a failure of this notebook - report it as one.")
acc.to_csv(OUT / "bench_C_selectivity_accuracy.csv")

### Test C, paired against the reference

The table above gives each tool its own interval against chance. That is **not**
a comparison between tools: "savant clears chance and Boltz-2 does not" is a
different claim from "savant beats Boltz-2", and only the second is what a reader
hears. Two overlapping intervals can easily hide a null paired difference.

These are paired on the same protein-pairs:

- **Cluster bootstrap by ligand.** Pairs from one ligand are not independent (a
  ligand measured against 4 proteins contributes 6 correlated pairs), so the
  resampling unit is the ligand, not the pair.
- **McNemar**, counting only the pairs where the two tools disagree - the exact
  test for two binary predictors scored on identical items.

In [ ]:
cen_hits = pd.DataFrame(cen_recs)
## bool-or-NaN records land as object dtype, and numpy refuses to subtract
## booleans ("use bitwise_xor"). Float is the right dtype for a hit rate anyway.
for c in cen_hits.columns:
    if c != "skeleton":
        cen_hits[c] = cen_hits[c].astype(float)

def paired_selectivity(tool, ref=REF, n_boot=N_BOOT, seed=SEED):
    if tool not in cen_hits or ref not in cen_hits:
        return {}
    d = (cen_hits[tool] - cen_hits[ref]).to_numpy(float)
    keep = np.isfinite(d)
    groups = cen_hits.skeleton.to_numpy()
    uniq = pd.unique(groups[keep])
    idx = {g: np.flatnonzero(keep & (groups == g)) for g in uniq}
    rng = np.random.default_rng(seed)
    draws = np.array([d[np.concatenate([idx[g] for g in rng.choice(uniq, len(uniq))])].mean()
                      for _ in range(n_boot)])
    both = cen_hits[[tool, ref]].dropna()
    b01 = int(((both[tool] == 1) & (both[ref] == 0)).sum())
    b10 = int(((both[tool] == 0) & (both[ref] == 1)).sum())
    return {"n_pairs": int(keep.sum()), "n_ligands": len(uniq),
            "mean_diff": float(np.nanmean(d)),
            "ci_low": float(np.percentile(draws, 2.5)),
            "ci_high": float(np.percentile(draws, 97.5)),
            "P_better": float((draws > 0).mean()),
            "wins": b01, "losses": b10,
            "mcnemar_p": float(binomtest(b01, b01 + b10, 0.5).pvalue) if b01 + b10 else np.nan}

sel_vs = table([{"tool": k, **paired_selectivity(k)}
                for k in cen_hits.columns if k not in ("skeleton", REF)],
               sort="mean_diff")
print(f"corrected selectivity, paired vs {REF}:")
display(sel_vs.round(3))
sel_vs.to_csv(OUT / "bench_C_paired_vs_ref.csv")
print("\nReport the paired row, not the difference between two single-tool CIs.")

In [ ]:
per_ligand = pd.DataFrame({k_: grouped_rank(multi, "skeleton", v, min_n=3)
                           for k_, v in SEL_TOOLS.items()})
if len(per_ligand):
    lig_sum = pd.DataFrame({
        "n_ligands": per_ligand.notna().sum(),
        "mean_spearman": per_ligand.mean(),
        "median_spearman": per_ligand.median(),
    }).sort_values("mean_spearman", ascending=False)
    display(lig_sum.round(3))
    print(f"\nligands with 3+ proteins: {(k >= 3).sum()}  "
          f"(Spearman over 3-4 points is coarse; the sign accuracy above is the "
          f"more reliable read)")
    per_ligand.to_csv(OUT / "bench_C_per_ligand.csv")
    lig_sum.to_csv(OUT / "bench_C_summary.csv")
else:
    print("no ligand has 3+ proteins with finite predictions")

## Test D - protein taxonomy

There is no taxonomy pipeline in this repo, so the family map is curated here,
explicitly, for the 20 benchmark targets. It is a plain dict: edit it rather than
working around it. Any UniProt ID not listed falls back to `unassigned` and is
reported separately rather than silently folded into another family.

Families with fewer than `MIN_STRATUM` rows get their `n` reported but no ranking
- a Spearman over 13 pairs in one family is not a finding.

In [ ]:
PROTEIN_FAMILY = {
    "P00915": "carbonic anhydrase", "P00918": "carbonic anhydrase",
    "Q16790": "carbonic anhydrase", "O43570": "carbonic anhydrase",
    "P31391": "GPCR class A",       "P37288": "GPCR class A",
    "P30518": "GPCR class A",
    "Q99640": "protein kinase",     "P30530": "protein kinase",
    "P68400": "protein kinase",     "Q01973": "protein kinase",
    "P37231": "nuclear receptor",
    "Q13224": "ion channel",
    "P42226": "transcription factor", "P28347": "transcription factor",
    "P01116": "small GTPase / PPI",   "Q07889": "small GTPase / PPI",
    "P62937": "isomerase",
    "P37840": "other",              "Q60492": "other",
}
df["protein_family"] = df.uniprot_id.map(PROTEIN_FAMILY).fillna("unassigned")
un = df[df.protein_family == "unassigned"].uniprot_id.unique()
if len(un):
    print(f"WARNING {len(un)} unmapped uniprot ids: {list(un)}  -> add them above")

print(df.groupby("protein_family").agg(
    n=("benchmark_id", "size"), targets=("uniprot_id", "nunique"),
    mean_pkd=(TRUTH, "mean")).sort_values("n", ascending=False).round(2).to_string())

In [ ]:
def stratified(data, key, use_within=True):
    '''Per-stratum metrics for every tool. within = mean per-target Spearman.'''
    rows = []
    for stratum, s in data.groupby(key):
        rec = {key: stratum, "n": len(s), "targets": s.uniprot_id.nunique(),
               "ligands": s.skeleton.nunique()}
        if len(s) >= MIN_STRATUM:
            for name, col in TOOLS.items():
                if name == "Constant mean":
                    continue
                rec[f"{name} | pooled_r"] = metrics(s[TRUTH], s[col]).get("pearson", np.nan)
                if use_within and name not in NO_WITHIN:
                    w = grouped_rank(s, "uniprot_id", col)
                    rec[f"{name} | within"] = w.mean() if len(w) else np.nan
        rows.append(rec)
    return pd.DataFrame(rows).set_index(key).sort_values("n", ascending=False)

fam = stratified(df, "protein_family")
display(fam.round(3))
fam.to_csv(OUT / "bench_D_protein_taxonomy.csv")
print(f"\nstrata with n < {MIN_STRATUM} are reported but not ranked")

### The carbonic anhydrase panel

Four isoforms, ~130 pairs, and a shared ligand set - the one family here with
enough targets and enough shared chemistry to say something specific. CA9 and
CA12 are the cancer-associated isoforms and selectivity over CA1/CA2 is the
actual design objective, so a tool that ranks compounds well inside each isoform
but cannot separate the isoforms is only half useful.

In [ ]:
ca = df[df.protein_family == "carbonic anhydrase"]
print(f"{len(ca)} pairs, {ca.uniprot_id.nunique()} isoforms, "
      f"{ca.skeleton.nunique()} ligands")
print(ca.groupby("protein_name")[TRUTH].agg(["size", "mean", "std"]).round(2).to_string())

rows = []
for name, col in TOOLS.items():
    w = pd.Series(dtype=float) if name in NO_WITHIN else grouped_rank(ca, "uniprot_id", col)
    m = metrics(ca[TRUTH], ca[col])
    rows.append({"tool": name, "pooled_r": m.get("pearson", np.nan),
                 "rmse": m.get("rmse", np.nan),
                 "within_isoform_spearman": w.mean() if len(w) else np.nan,
                 "n_isoforms": len(w)})
ca_t = table(rows, sort="within_isoform_spearman")
display(ca_t.round(3))
ca_t.to_csv(OUT / "bench_D_carbonic_anhydrase.csv")

## Test E - ligand taxonomy

Also built here, offline, with RDKit - no ClassyFire call, so it reruns
identically on a cluster with no network. Each ligand gets **one** primary class
by priority (a sulfonamide that is also a carboxylic acid is filed as a
sulfonamide, since that is the group driving carbonic anhydrase binding).

Three independent ligand axes, because chemotype alone will be dominated by the
CA sulfonamides:

- **chemotype** - SMARTS classes below
- **size** - heavy-atom-count tertiles
- **novelty** - `novelty_tier`, distance to the nearest training ligand

In [ ]:
from rdkit import Chem, RDLogger
from rdkit.Chem import Descriptors, rdMolDescriptors
RDLogger.DisableLog("rdApp.*")

## priority order: first match wins
SMARTS = [
    ("macrocycle",          None),                                  # ring size >= 12
    ("peptide-like",        "[NX3][CX3](=[OX1])[CX4][NX3][CX3]=[OX1]"),
    ("primary sulfonamide", "[SX4](=[OX1])(=[OX1])[NX3;H2]"),
    ("sulfonamide",         "[SX4](=[OX1])(=[OX1])[NX3]"),
    ("carboxylic acid",     "[CX3](=[OX1])[OX2H1]"),
    ("fused heteroaromatic", "[a;R2]"),
    ("halo-aromatic",       "[c][F,Cl,Br,I]"),
    ("basic amine",         "[NX3;H2,H1;!$(N[#6]=[O,N,S])]"),
]
PATT = [(n, Chem.MolFromSmarts(s) if s else None) for n, s in SMARTS]

def classify(smi):
    m = Chem.MolFromSmiles(smi)
    if m is None:
        return "unparseable"
    if max((len(r) for r in m.GetRingInfo().AtomRings()), default=0) >= 12:
        return "macrocycle"
    for name, patt in PATT:
        if patt is not None and m.HasSubstructMatch(patt):
            return name
    return "other"

uniq = df.ligand_smiles.drop_duplicates()
cls = {s: classify(s) for s in uniq}
heavy = {s: (Chem.MolFromSmiles(s).GetNumHeavyAtoms()
             if Chem.MolFromSmiles(s) else np.nan) for s in uniq}
df["ligand_class"] = df.ligand_smiles.map(cls)
df["heavy_atoms"]  = df.ligand_smiles.map(heavy)
df["ligand_size"]  = pd.qcut(df.heavy_atoms, 3,
                             labels=["small", "medium", "large"]).astype(str)
if "novelty_tier" not in df.columns and "max_train_ligand_tanimoto" in df.columns:
    df["novelty_tier"] = pd.cut(df.max_train_ligand_tanimoto,
                                [-0.01, 0.25, 0.35, 1.0],
                                labels=["distant", "moderate", "near"]).astype(str)

print(df.groupby("ligand_class").agg(
    n=("benchmark_id", "size"), ligands=("skeleton", "nunique"),
    mean_pkd=(TRUTH, "mean")).sort_values("n", ascending=False).round(2).to_string())
print(f"\nheavy atoms: {df.heavy_atoms.min():.0f}-{df.heavy_atoms.max():.0f}, "
      f"median {df.heavy_atoms.median():.0f}")

In [ ]:
for axis, fname in [("ligand_class", "bench_E_ligand_chemotype.csv"),
                    ("ligand_size",  "bench_E_ligand_size.csv"),
                    ("novelty_tier", "bench_E_ligand_novelty.csv")]:
    if axis not in df.columns:
        continue
    t = stratified(df, axis)
    print(f"\n=== {axis} ===")
    display(t.round(3))
    t.to_csv(OUT / fname)

## Test F - savant vs a model trained on the target itself

Tests A-E ask how savant compares to other *general* tools. This one asks the
question that decides whether the tool has a place: when a chemist already has
data on the target, is a general model worth using at all?

The headline is not who wins on average - it is **where the crossover is**. A
per-target model is only as good as the data behind it, so the honest output is
the gap as a function of how many prior ligands the target has. Three outcomes,
all publishable:

- **The baselines win everywhere** - savant's value is then targets with little
  or no data, and the framing narrows to exactly that.
- **Savant wins everywhere** - a strong claim: one model competitive with
  per-target models, with no target-specific fitting.
- **A crossover** - the most likely and most useful: savant below some data
  threshold, local models above it. That threshold is a deliverable in itself.

Read the per-target table, not the mean. A mean over 20 targets whose training
sets range over orders of magnitude conflates the two regimes this test exists
to separate.

In [ ]:
if df.qsar_pk.notna().sum() == 0:
    print("no per-target baselines were computed; skipping test F")
else:
    LOCAL = [t for t in ("QSAR per-target", "Tanimoto kNN") if t in TOOLS]
    SAV = [t for t in TOOLS if t.startswith("Savant")]
    cols = SAV + LOCAL + [REF]

    f_local = pd.DataFrame({t: grouped_rank(df, "uniprot_id", TOOLS[t]) for t in cols})
    f_local.insert(0, "n_train", pd.Series(QSAR_N_TRAIN))
    f_local.insert(1, "n_bench", df.groupby("uniprot_id").size())
    f_local = f_local.sort_values("n_train")
    display(f_local.round(3))
    f_local.to_csv(OUT / "bench_F_per_target_vs_local.csv")

    print("\nmean within-target Spearman (targets where every column is defined):")
    display(f_local[cols].dropna().mean().sort_values(ascending=False).round(3).to_frame("mean"))

In [ ]:
if df.qsar_pk.notna().sum():
    BEST_SAV = max(SAV, key=lambda t: f_local[t].mean())
    print(f"best savant variant on this test: {BEST_SAV}\n")

    rows = []
    rng_f = np.random.default_rng(SEED)
    for t in LOCAL + [REF]:
        d = (f_local[BEST_SAV] - f_local[t]).dropna()
        if len(d) < 3:
            continue
        draws = d.to_numpy()[rng_f.integers(0, len(d), size=(N_BOOT, len(d)))].mean(axis=1)
        rows.append({"vs": t, "n_targets": len(d), "savant_minus": float(d.mean()),
                     "ci_low": float(np.percentile(draws, 2.5)),
                     "ci_high": float(np.percentile(draws, 97.5)),
                     "savant_wins": int((d > 0).sum()),
                     "wilcoxon_p": float(wilcoxon(d).pvalue) if len(d) >= 5 else np.nan})
    vs_local = pd.DataFrame(rows).set_index("vs")
    display(vs_local.round(3))
    vs_local.to_csv(OUT / "bench_F_vs_local.csv")

    ## the crossover: does the local model's advantage grow with its data?
    g = f_local.dropna(subset=[BEST_SAV] + LOCAL)
    print("\ndoes more target data favour the local model?")
    for t in LOCAL:
        r = spearmanr(g.n_train, g[t] - g[BEST_SAV])
        print(f"  ({t} - savant) vs n_train: spearman {r.correlation:+.3f} "
              f"(p={r.pvalue:.3f}); positive means the local model pulls ahead "
              f"as target data grows")
    med = g.n_train.median()
    lo, hi = g[g.n_train <= med], g[g.n_train > med]
    print(f"\nsplit at the median training-set size ({med:.0f} ligands):")
    for t in [BEST_SAV] + LOCAL + [REF]:
        print(f"  {t:18s} data-poor half {lo[t].mean():+.3f} | "
              f"data-rich half {hi[t].mean():+.3f}")

In [ ]:
if df.qsar_pk.notna().sum():
    fig, ax = plt.subplots(figsize=(8, 5))
    style = {BEST_SAV: ("#2a78d6", "o"), REF: ("#eb6834", "s")}
    for t in LOCAL:
        style[t] = ("#eda100" if t == "QSAR per-target" else "#639922", "^")
    for t, (c, m) in style.items():
        if t in f_local:
            ax.scatter(f_local.n_train.clip(lower=1), f_local[t], s=46, color=c, marker=m,
                       label=t, alpha=.85)
    ax.set_xscale("log")
    ax.axhline(0, color="#888780", lw=.8)
    ax.set(xlabel="prior ligands for this target in training (log scale)",
           ylabel="within-target Spearman",
           title="Test F - a general model vs models fit on the target itself")
    ax.legend(fontsize=8, frameon=False)
    ax.spines[["top", "right"]].set_visible(False)
    fig.tight_layout()
    fig.savefig(OUT / "bench_F_crossover.png", dpi=150, bbox_inches="tight")
    plt.show()

## Classification

Two thresholds, because one of them is uninformative on this benchmark. At the
trained cut of pKd 6.0 about 89% of the benchmark is a positive, so accuracy and
F1 mostly measure the base rate; ROC and PR still mean something but a PR AUC near
0.9 is what the prevalence alone buys. The 7.0 cut splits this set close to
even and is the honest comparison - it is simply not the cut the models were
trained on, so read it as ranking quality, not calibration.

Every tool's continuous score is used as the ranking; the dedicated probability
outputs are scored separately where they exist.

In [ ]:
from sklearn.metrics import (accuracy_score, average_precision_score, f1_score,
                             roc_auc_score)

rows = []
for cut in BINDER_CUTS:
    truth_bin = (df[TRUTH] > cut).astype(int)
    for name, col in TOOLS.items():
        if name == "Constant mean":
            continue
        m = np.isfinite(df[col])
        if truth_bin[m].nunique() < 2:
            continue
        rows.append({"tool": name, "cut": cut, "n": int(m.sum()),
                     "positive_rate": float(truth_bin[m].mean()),
                     "roc_auc": roc_auc_score(truth_bin[m], df[col][m]),
                     "pr_auc": average_precision_score(truth_bin[m], df[col][m]),
                     "source": "regression score"})
    for name, col in CLF_COLS.items():
        m = np.isfinite(df[col])
        if truth_bin[m].nunique() < 2:
            continue
        pred = df[col][m] >= 0.5
        rows.append({"tool": name, "cut": cut, "n": int(m.sum()),
                     "positive_rate": float(truth_bin[m].mean()),
                     "roc_auc": roc_auc_score(truth_bin[m], df[col][m]),
                     "pr_auc": average_precision_score(truth_bin[m], df[col][m]),
                     "accuracy": accuracy_score(truth_bin[m], pred),
                     "f1": f1_score(truth_bin[m], pred, zero_division=0),
                     "source": "classifier probability"})
clf = pd.DataFrame(rows).sort_values(["cut", "roc_auc"], ascending=[True, False])
display(clf.round(3))
clf.to_csv(OUT / "bench_classification.csv", index=False)

## Runtime

The cost argument is savant's main practical differentiator, so it has to be
measured rather than asserted.

**It runs in a subprocess with `CUDA_VISIBLE_DEVICES=""`.** Setting that after
torch has already been imported does nothing, and this notebook has almost
certainly imported torch above to score the models - so an in-process "CPU"
measurement on a GPU node would silently time the GPU. A clean subprocess is the
only honest way to get the number.

**What gets timed, and why separately.** Savant's cost splits into a part that
depends on the protein and a part that depends on the ligand. In a screening run
you embed the target **once** and reuse it for every compound, so the marginal
cost of an additional ligand excludes the protein encoder entirely. Boltz-2
co-folds each protein-ligand complex and cannot amortise anything. That
asymmetry, not the per-pair number, is the argument - so the cell reports both.

Set `BOLTZ_SEC_PER_PAIR` from your own Boltz-2 logs to get the comparison. Leave
it `None` and that row is omitted rather than guessed.

In [ ]:
import os, subprocess, tempfile, textwrap
import torch

RUNTIME_N_PROTEINS = 8       # unique proteins to time the ESM encoder on
RUNTIME_N_LIGANDS  = 64      # unique ligands to time the ligand path on
RUNTIME_MODEL      = "esm650"
RUNTIME_DIR        = MODEL_SETS[""]     # time the current generation
BOLTZ_SEC_PER_PAIR = None    # <- fill in from your Boltz-2 run to enable the comparison

script = textwrap.dedent(f'''
    import json, os, sys, time
    import numpy as np
    import torch

    ## CUDA_VISIBLE_DEVICES="" is not honoured by every torch build - on some it
    ## still reports a device, and the "CPU" row would silently time the GPU.
    ## Patching the probe savant actually calls is deterministic.
    if os.environ.get("SAVANT_TIMING_FORCE_CPU") == "1":
        torch.cuda.is_available = lambda: False

    sys.path.insert(0, r"{SAVANT_PATH}")
    from savant import Savant, encoders, features

    seqs   = json.loads(sys.argv[1])
    smiles = json.loads(sys.argv[2])
    sv = Savant(model="{RUNTIME_MODEL}", models_dir=r"{RUNTIME_DIR}")

    def clock(fn):
        t = time.perf_counter(); out = fn(); return out, time.perf_counter() - t

    CB = encoders.CHEMBERTA
    _, t_warm = clock(lambda: encoders.encode(smiles[:2], CB, 2))
    mols, t_parse = clock(lambda: features.parse_smiles(smiles))
    _, t_cb   = clock(lambda: encoders.encode(smiles, CB, 64))
    _, t_desc = clock(lambda: features.descriptors_named(mols, sv.rdkit_columns))
    _, t_fp   = clock(lambda: features.ecfp4(mols))
    _, t_esm  = clock(lambda: encoders.encode(seqs, sv.esm_model, 8, half=False,
                                              max_length=1024))
    ## predict_from, not the boosters directly: it is the production call and it
    ## handles a bundle with no classifier (sv.classifier is None on the
    ## regressor-only models, where calling .inplace_predict would raise)
    X = np.zeros((len(smiles), sv.n_features_in_), dtype=np.float32)
    _, t_pred = clock(lambda: sv.predict_from(X))

    json.dump({{"device": "cuda" if torch.cuda.is_available() else "cpu",
               "threads": torch.get_num_threads(),
               "n_prot": len(seqs), "n_lig": len(smiles),
               "esm_per_protein": t_esm / len(seqs),
               "chemberta_per_ligand": t_cb / len(smiles),
               "rdkit_per_ligand": (t_parse + t_desc + t_fp) / len(smiles),
               "predict_per_pair": t_pred / len(smiles),
               "warmup": t_warm}}, open(sys.argv[3], "w"))
''')

prot = df.target_sequence.drop_duplicates().head(RUNTIME_N_PROTEINS).tolist()
lig  = df.ligand_smiles.drop_duplicates().head(RUNTIME_N_LIGANDS).tolist()

## preflight: fail here with a clear reason rather than inside a subprocess
_bundle = Path(RUNTIME_DIR) / f"savant_full_{RUNTIME_MODEL}.pkl"
_vdir = Path(RUNTIME_DIR) / RUNTIME_MODEL
print(f"timing {RUNTIME_MODEL} from {RUNTIME_DIR}")
print(f"  bundle {_bundle.name}: {'found' if _bundle.exists() else 'MISSING'}"
      f"   raw dir {RUNTIME_MODEL}/: {'found' if _vdir.exists() else 'missing'}")
print(f"  {len(prot)} proteins, {len(lig)} ligands")

RUNTIME = {}
if not (_bundle.exists() or _vdir.exists()):
    print(f"no model for {RUNTIME_MODEL} in {RUNTIME_DIR} - skipping timing")
    RUNTIME_DEVICES = []
else:
    RUNTIME_DEVICES = [("CPU", True), ("GPU", False)]
for label, hide_gpu in RUNTIME_DEVICES:
    if not hide_gpu and not torch.cuda.is_available():
        continue
    with tempfile.TemporaryDirectory() as td:
        sp, jp = Path(td) / "t.py", Path(td) / "t.json"
        sp.write_text(script)
        env = dict(os.environ)
        env["SAVANT_TIMING_FORCE_CPU"] = "1" if hide_gpu else "0"
        if hide_gpu:
            env["CUDA_VISIBLE_DEVICES"] = ""
        r = subprocess.run([sys.executable, str(sp), json.dumps(prot),
                            json.dumps(lig), str(jp)],
                           capture_output=True, text=True, env=env)
        if r.returncode != 0 or not jp.exists():
            print(f"\n{'!' * 70}\n{label} TIMING FAILED (exit {r.returncode})")
            print("stderr:")
            print(r.stderr[-3000:] or "(empty)")
            if r.stdout.strip():
                print("stdout:")
                print(r.stdout[-1000:])
            print("common causes: the model bundle is not in RUNTIME_DIR; the "
                  "transformers cache is unreachable on a compute node with no "
                  "network (set HF_HOME / TRANSFORMERS_OFFLINE=1); or not enough "
                  "RAM to hold ESM2-650M in fp32 on CPU (~2.6 GB).")
            print("!" * 70)
            continue
        RUNTIME[label] = json.loads(jp.read_text())
    dev = RUNTIME[label]["device"]
    print(f"{label}: measured on device={dev}, "
          f"{RUNTIME[label]['threads']} torch threads")
    if label == "CPU" and dev != "cpu":
        print("  WARNING: asked for CPU but the run reported "
              f"device={dev} - do not quote this as a CPU number")

In [ ]:
## tolerate the timing cell being skipped or failing - it is the slowest cell
## here and the rest of the notebook does not depend on it
RUNTIME = globals().get("RUNTIME", {})
if not RUNTIME:
    print("no timings collected (timing cell skipped or failed)")
else:
    rows = []
    for label, t in RUNTIME.items():
        per_lig = t["chemberta_per_ligand"] + t["rdkit_per_ligand"] + t["predict_per_pair"]
        rows.append({
            "device": label,
            "esm per protein (s)": t["esm_per_protein"],
            "per extra ligand (s)": per_lig,
            "cold single pair (s)": t["esm_per_protein"] + per_lig,
            "500 pairs, 20 targets (s)": 20 * t["esm_per_protein"] + 500 * per_lig,
            "1 target x 100k ligands (h)": (t["esm_per_protein"] + 100_000 * per_lig) / 3600,
        })
    rt = pd.DataFrame(rows).set_index("device")
    display(rt.round(4))
    rt.to_csv(OUT / "bench_runtime.csv")

    cpu = RUNTIME.get("CPU")
    if cpu:
        per_lig = (cpu["chemberta_per_ligand"] + cpu["rdkit_per_ligand"]
                   + cpu["predict_per_pair"])
        print(f"\nSavant, CPU only:")
        print(f"  one pair, cold          {cpu['esm_per_protein'] + per_lig:8.3f} s")
        print(f"  each additional ligand  {per_lig:8.4f} s   "
              f"({cpu['esm_per_protein'] / per_lig:.0f}x cheaper than the first)")
        if BOLTZ_SEC_PER_PAIR:
            b = BOLTZ_SEC_PER_PAIR
            print(f"\nBoltz-2 at {b:.1f} s/pair (GPU, your measurement):")
            print(f"  screening 1 target x 100k ligands")
            print(f"    savant  {(cpu['esm_per_protein'] + 100_000*per_lig)/3600:10.2f} CPU-hours")
            print(f"    boltz-2 {100_000*b/3600:10.2f} GPU-hours")
            print(f"    ratio   {(100_000*b)/(cpu['esm_per_protein'] + 100_000*per_lig):9.0f}x")
            print("  (GPU-hours and CPU-hours are not the same unit - state both, "
                  "and price them if you want a cost claim)")
        else:
            print("\nSet BOLTZ_SEC_PER_PAIR from your Boltz-2 logs for the comparison.")

## Charts

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4.6))
order = [t for t in TOOLS if t != "Constant mean"]
colors = ["#85b7eb" if t.startswith("Savant-v2") else
          "#2a78d6" if t.startswith("Savant") else
          "#eb6834" if t == "Boltz-2" else
          "#1baf7a" if "PRODIGY" in t else
          "#4a3aa7" if t == "PLAPT" else
          "#eda100" if t in ("QSAR per-target", "Tanimoto kNN") else
          "#888780" for t in order]

ax = axes[0]
ax.bar(range(len(order)), [pooled.loc[t, "pearson"] for t in order], color=colors)
ax.axhline(0, color="#888780", lw=0.8)
ax.set_title("A. pooled Pearson\n(blends both abilities)", fontsize=10)

ax = axes[1]
ax.bar(range(len(order)), [summary.loc[t, "mean_spearman"] if t in summary.index
                           else np.nan for t in order], color=colors)
ax.axhline(0, color="#888780", lw=0.8)
ax.set_title("B. within-target Spearman\n(rank a compound series)", fontsize=10)

ax = axes[2]
sa = [acc.loc[t, "corrected_accuracy"] if t in acc.index else np.nan for t in order]
ax.bar(range(len(order)), sa, color=colors)
ax.axhline(0.5, color="#e34948", lw=1, ls="--", label="chance")
ax.set_ylim(0.3, 1.0)
ax.legend(fontsize=8)
ax.set_title("C. selectivity, base-rate corrected\n(does THIS compound prefer it)",
             fontsize=10)

for ax in axes:
    ax.set_xticks(range(len(order)))
    ax.set_xticklabels(order, rotation=35, ha="right", fontsize=8)
    ax.spines[["top", "right"]].set_visible(False)
fig.tight_layout()
fig.savefig(OUT / "bench_summary.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
present = [t for t in ("Savant esm650", "Savant esm8", "Boltz-2") if t in TOOLS]
fig, axes = plt.subplots(1, len(present), figsize=(4.6 * len(present), 4.4),
                         squeeze=False)
lims = (df[TRUTH].min() - 0.4, df[TRUTH].max() + 0.4)
fams = df.protein_family.unique()
cmap = dict(zip(fams, plt.cm.tab10.colors))
for ax, t in zip(axes[0], present):
    col = TOOLS[t]
    for fam_name in fams:
        s = df[df.protein_family == fam_name]
        ax.scatter(s[TRUTH], s[col], s=14, alpha=0.75, color=cmap[fam_name], label=fam_name)
    ax.plot(lims, lims, color="#888780", lw=0.9, ls="--")
    ax.set(xlim=lims, ylim=lims, xlabel="true pKd", title=t)
    ax.spines[["top", "right"]].set_visible(False)
axes[0][0].set_ylabel("predicted pKd")
axes[0][-1].legend(fontsize=7, loc="upper left", frameon=False)
fig.suptitle("Predicted vs true, coloured by protein family "
             "(a horizontal band = no between-target signal)", y=1.02)
fig.tight_layout()
fig.savefig(OUT / "bench_scatter_by_family.png", dpi=150, bbox_inches="tight")
plt.show()

## Save

In [ ]:
keep = [c for c in df.columns if not c.startswith("Unnamed")]
df[keep].to_csv(OUT / "bench_scored.csv", index=False)

## to_string in a fenced block, not to_markdown: to_markdown needs `tabulate`,
## which is an optional pandas dependency and is not in the eval env
def block(t):
    return "```\n" + t.round(3).to_string() + "\n```"

lines = [f"# v3 benchmark - {len(df)} pairs, {df.uniprot_id.nunique()} targets, "
         f"{df.skeleton.nunique()} ligands", "",
         f"source: {BENCH.name}", "",
         "## A. pooled", block(pooled), "",
         f"constant-mean RMSE {const_rmse:.3f} - a tool whose RMSE is above this "
         "is not beating 'predict the average'", "",
         "## Between vs within decomposition", block(decomp), "",
         "## B. one protein, many ligands", block(summary), "",
         f"paired vs {REF}", block(vs), "",
         "## C. one ligand, many proteins (selectivity)", block(acc), "",
         "read corrected_accuracy; raw_accuracy is inflated by target base rates", "",
         f"paired vs {REF}", block(sel_vs), "",
         "## D. protein taxonomy", block(fam), "",
         "## E. ligand chemotype", block(stratified(df, "ligand_class"))]
if df.qsar_pk.notna().sum():
    lines += ["", "## F. vs models trained on the target itself", block(f_local), "",
              f"paired, best savant variant ({BEST_SAV})", block(vs_local), "",
              "n_train is how many prior ligands that target had in training; "
              "read the crossover, not the mean"]
(OUT / "bench_summary.md").write_text("\n".join(lines), encoding="utf-8")

print("wrote:")
for f in sorted(OUT.glob("bench_*")):
    print(f"  {f.name}")